# QuEL-1 continuous-wave output

This notebook shows how to use `Quel1BackendController` to start and stop a long-running continuous-wave output on a QuEL-1 AWG channel.

The CW methods are QuEL-1-family-specific debugging tools. They are not part of the common `BackendController` contract, and they are not exposed through `Measurement` or `Experiment`.

The code is guarded by `RUN_HARDWARE` so the notebook can be opened safely. Set the box and clockmaster values for your setup, then set `RUN_HARDWARE = True` only when the selected box, port, and channel are connected and ready for RF output.

In [ ]:
from qubex.backend.quel1 import Quel1BackendController

RUN_HARDWARE = False

BOX_NAME = "S173R"
BOX_IPADDR_WSS = "10.1.0.x"
BOXTYPE = "quel1se-riken8"
CLOCKMASTER_IPADDR = "10.3.0.x"
PORT = 1
CHANNEL = 0

AWG_FREQ_HZ = 7_812_500.0 * 20
AMPLITUDE = 0.2

`AWG_FREQ_HZ` is the AWG-baseband frequency, not the final RF frequency. CW is generated by repeating one 128 ns waveform chunk, so the chunk must contain an integer number of AWG cycles. The frequency grid is therefore `1 / 128 ns = 7.8125 MHz`; use `7_812_500.0 * N` for integer `N`.

## Start CW with the current output settings

By default, `start_continuous_wave()` preserves the current output path. It does not call `config_port()` and it does not update FNCO. `awg_freq_hz` still changes the generated AWG waveform. This avoids changing the port phase reference as a side effect of starting CW output.

In [ ]:
controller = Quel1BackendController()

if RUN_HARDWARE:
    controller.define_clockmaster(ipaddr=CLOCKMASTER_IPADDR)
    controller.define_box(
        box_name=BOX_NAME,
        ipaddr_wss=BOX_IPADDR_WSS,
        boxtype=BOXTYPE,
    )
    controller.connect(box_names=BOX_NAME)
    print(f"Connected to {BOX_NAME}")
else:
    print("Set RUN_HARDWARE = True to connect to hardware.")

In [ ]:
if RUN_HARDWARE:
    cw_config = controller.start_continuous_wave(
        box_name=BOX_NAME,
        port=PORT,
        channel=CHANNEL,
        awg_freq_hz=AWG_FREQ_HZ,
        amplitude=AMPLITUDE,
    )
    print(cw_config)
else:
    print("Set RUN_HARDWARE = True to start CW output on hardware.")

When CW starts, Qubex logs the LO, CNCO, FNCO, AWG, `FNCO + AWG`, and computed RF output frequency in GHz at `INFO` level. If `abs(FNCO + AWG)` is greater than 800 MHz, Qubex emits a warning because aliasing may appear. The warning does not stop the output.

## Start CW while updating output settings

Use `configure_port=True` only when the CW start should also update output settings. Concrete LO/CNCO/FNCO, sideband, VATT, full-scale current, or RF switch values are rejected unless `configure_port=True` is set.

In [ ]:
if RUN_HARDWARE:
    controller.stop_all_continuous_waves()
    cw_config = controller.start_continuous_wave(
        box_name=BOX_NAME,
        port=PORT,
        channel=CHANNEL,
        awg_freq_hz=0.0,
        amplitude=0.1,
        configure_port=True,
        lo_freq_hz=10_500_000_000,
        cnco_freq_hz=2_250_000_000,
        fnco_freq_hz=750_000_000,
        sideband="U",
        vatt=2048,
        fullscale_current=39000,
        rfswitch="pass",
    )
    print(cw_config)

## Stop CW output

Stop one remembered output by the same box, port, and channel. `stop_continuous_wave()` returns `True` when the controller had an active CW task for that key.

In [ ]:
if RUN_HARDWARE:
    stopped = controller.stop_continuous_wave(
        box_name=BOX_NAME,
        port=PORT,
        channel=CHANNEL,
    )
    print(stopped)

Use `stop_all_continuous_waves()` before disconnecting or before changing to a different CW setup. `disconnect()` also attempts to stop remembered CW tasks on a best-effort basis.

In [ ]:
if RUN_HARDWARE:
    controller.stop_all_continuous_waves()
    controller.disconnect()